# Chapter 13: Agentic RAG
## Topic 4: Testing Against the New-Product-Name Failure Pattern (Smart Saver FD)

**One notebook. Theory + Code together.**


## Part A: Theory

### 1. Concept, Intuition, and Why This Exists

- Chapter 9 Topic 8 already proved something important: once retrieval is triggered, the RAG pipeline correctly grounds an answer about "Smart Saver FD" in real, ingested data, producing a 0/2-vs-2/2 correct-figure result comparing ungrounded and grounded responses. That test, however, explicitly *assumed* retrieval was already triggered — it deliberately deferred the question of whether the agent would *decide* to retrieve in the first place, calling that "Chapter 13's separate, later concern." This topic is that deferred concern, finally addressed directly.
- This chapter's Topics 1-3 built exactly the machinery this test needs: Topic 1 established agentic RAG's control-structure difference from static RAG, Topic 2 established confidence-based triggering as the right general principle, and Topic 3 built the actual "check catalog only if uncertain" decision step as a concrete instance of that principle. This topic's test validates that this specific, concrete decision step — not just the downstream grounded-generation quality Chapter 9 Topic 8 already validated — actually works correctly for the exact failure pattern it was designed to catch.
- The precise thing being tested here, stated carefully: not "does RAG produce a correct answer once triggered" (Chapter 9 Topic 8's already-answered question) but "does the agent's own triggering decision correctly fire when a genuinely novel product name appears" — a test of the *decision*, using Topic 3's specific pattern, applied to the specific, maximally-clean test case Chapter 9 Topic 8 originally designed for a different, narrower purpose.


### 2. Internal Working — Step by Step

**The complete, end-to-end test this topic builds, combining every relevant piece from this notebook series:**

1. **Start from Chapter 9 Topic 8's exact test design principle**: an invented, genuinely novel product name ("Smart Saver FD") with specific, fabricated terms that could not exist in any model's training data, since the product was invented specifically for this testing purpose — this is what makes the test's failure mode for an ungrounded model *structurally guaranteed*, not just probable.
2. **Apply Topic 3's detection-and-trigger decision step to a query mentioning this invented product**, checking specifically whether the decision step correctly identifies "Smart Saver FD" as a named product warranting a catalog check — this is testing the *triggering* logic in isolation, independent of whether the underlying `lookup_product_catalog` tool (Chapter 10 Topic 6) or the retrieval pipeline (Chapter 7) behave correctly, exactly the layered-attribution discipline this notebook series has repeatedly required.
3. **If triggering succeeds, confirm the full pipeline behaves exactly as Chapter 9 Topic 8 already validated**: the catalog lookup (or, in the broader RAG case, `search_knowledge_base` retrieval) correctly finds the newly-added product information, and the generated answer correctly cites the real, specific figures rather than a generic, hallucinated approximation.
4. **The complete, end-to-end validation this topic performs is genuinely new relative to Chapter 9 Topic 8**: that catalog lookup correctly identifies the product (Chapter 10 Topic 6's found/not-found logic, applied to genuinely novel content added to the catalog for this test) *and* that the agent's own decision to call the lookup tool at all was triggered correctly by Topic 3's pattern — testing the full chain from "novel product mentioned" through "agent decides to check" through "check finds the right data" through "answer correctly cites it," rather than any single link in isolation.


### 3. How This Is Implemented in This Project

- This test reuses Chapter 9 Topic 8's exact invented product and its specific, fabricated figures — no need to invent a new test case, since Smart Saver FD's specific terms (interest rate, tenure, minimum deposit) were already carefully designed as a clean, novel test case in that earlier topic, and this topic's contribution is testing a genuinely different part of the pipeline (the triggering decision) using that same, already-validated test artifact.
- This test also reuses Topic 3's exact `detect_named_product()` and `should_check_catalog()` functions unchanged — validating that Topic 3's specific pattern, built and unit-tested there against a general labeled set, also correctly handles this specific, maximally-clean edge case (a product genuinely absent from any conceivable training data) as one particular instance of its broader, already-measured 100% accuracy.
- Following Chapter 10 Topic 7's test-suite discipline, this test should be added as a standing, re-runnable regression case — mirroring exactly Chapter 9 Topic 8's own recommendation that its proof-of-concept be re-run whenever the pipeline changes — since a future prompt, schema, or model change could regress either the triggering decision (this topic's focus) or the downstream grounded-generation quality (Chapter 9 Topic 8's focus) independently.


### 4. Real-World Issues, Edge Cases, Debugging, Monitoring, Scaling, Latency, Cost, Security, Deployment

- **This test's validity depends on Smart Saver FD genuinely remaining absent from whatever underlying data source is used**, exactly the same fragility Chapter 9 Topic 8 already noted — if this fictional product name happened to coincidentally match a real product that entered general use or training data after this test was designed, the test's clean, guaranteed-failure-mode property for the ungrounded case would be compromised, requiring a different or refreshed invented product name.
- **A failure at this specific test could originate in either of two genuinely separate layers, and correctly attributing which one matters for the fix**: if the triggering decision (Topic 3's pattern) fails to recognize the invented product name at all, that's a detection-step problem (perhaps the name's specific phrasing doesn't match the detection logic's pattern-matching, Chapter 9 Topic 2's Hinglish/vocabulary-mismatch discussion applied to product-name variants); if triggering succeeds but the catalog lookup or generation still produces a wrong answer, that's the layer Chapter 9 Topic 8 already validated, meaning a *regression* in that previously-working part of the pipeline, not a new triggering problem.
- **This test is deliberately narrow and clean, and shouldn't be mistaken for comprehensive coverage of every possible triggering edge case** — it validates one specific, maximally-clear scenario well; Chapter 10 Topic 7's broader, larger test suite (covering ambiguous, informal, and boundary cases beyond this one clean example) remains necessary for genuinely comprehensive triggering-reliability confidence, exactly as Chapter 9 Topic 8 itself cautioned against over-generalizing from a single clean proof point.
- **Monitoring:** if this specific test is added as a standing regression check (per Chapter 10 Topic 7's discipline), tracking its pass/fail status over time, specifically flagging *which* layer failed when it does fail (triggering vs downstream grounding), gives immediate, specific diagnostic value the very first time a future change breaks either part of this chain.


### 5. Design Decisions, Trade-offs, and Real-Time Dilemmas

- **Testing the triggering decision and the downstream grounding separately, in the same test run, vs treating "did the final answer come out correct" as the only pass/fail signal:** testing separately (this topic's approach) gives specific, actionable diagnostic information the moment something breaks — exactly which layer regressed — while a single end-to-end pass/fail signal alone would only reveal *that* something broke, not *where*, reproducing the same coarse-grained-failure-category problem this notebook series has repeatedly warned against.
- **Reusing Chapter 9 Topic 8's exact invented product vs designing a fresh one for this specific triggering test:** reuse (this topic's approach) is simpler and avoids proliferating multiple, slightly-different fictional test products across the notebook series, but does mean this one invented product now needs to remain "novel" and untested-elsewhere indefinitely for both tests to stay valid — a small, acceptable coordination cost given the benefit of a single, consistent, well-understood test artifact.
- **How much this specific test should be trusted as ongoing evidence vs how much it should trigger periodic refresh with a new invented product name:** since language models are periodically retrained or updated, and a sufficiently long-lived fictional product name could theoretically leak into future training data through this very type of testing documentation being published or shared, periodically confirming (or refreshing) the test's "genuinely novel" property is a reasonable, low-cost precaution rather than assuming it holds indefinitely without re-verification.


### 6. Alternatives and When to Use Each

- **This topic's layered test (triggering decision tested separately from downstream grounding, both using Chapter 9 Topic 8's clean invented-product design):** the right approach for validating this project's specific agentic-triggering pattern with maximum diagnostic clarity when something fails.
- **A single, coarser end-to-end pass/fail test with no layer separation:** simpler to build, but provides much weaker diagnostic value when it fails — appropriate only as an initial, minimal smoke test before investing in the more diagnostic, layered version this topic actually builds.
- **A broader, larger battery of triggering test cases beyond this one clean example (Chapter 10 Topic 7's fuller test-suite discipline):** necessary for comprehensive triggering-reliability confidence — this topic's specific test is a strong, clean point-validation, not a substitute for that broader coverage.


### 7. Common Mistakes and Production Failures

- Treating this test's success as validating comprehensive triggering reliability generally, rather than recognizing it as one clean, specific, narrow validation point that still requires Chapter 10 Topic 7's broader test-suite coverage for full confidence.
- Not separating the triggering-decision layer from the downstream grounding layer when this test fails, missing the specific, actionable diagnostic information layered testing is specifically designed to provide.
- Allowing the invented product name's "genuine novelty" property to go unverified indefinitely, risking the test silently losing its clean, guaranteed-failure-mode validity if the name were ever to coincidentally enter real usage or training data.
- Not adding this test as a standing, re-runnable regression check (per Chapter 10 Topic 7's discipline), leaving it as a one-time validation that could silently regress without detection after a future prompt, schema, or model change.


### 8. Lead-Level Interview Questions

**Basic**

- Q: How does this topic's test differ from Chapter 9 Topic 8's original Smart Saver FD proof-of-concept?
  A: Chapter 9 Topic 8 validated that RAG, once retrieval is triggered, produces a correctly grounded answer for genuinely novel content — it explicitly assumed the triggering decision was already made. This topic tests the triggering decision itself: does the agent correctly decide to check the catalog when a genuinely novel product name appears, using the specific pattern built in Topic 3, rather than assuming that decision happens correctly.

- Q: Why is reusing the exact same invented product (rather than designing a new one) a reasonable choice for this test?
  A: Chapter 9 Topic 8's invented product and its specific figures were already carefully designed as a clean test case — genuinely absent from any training data, with distinctive enough figures that a correct answer couldn't be a lucky coincidence. Reusing it avoids unnecessarily proliferating multiple fictional test products, while still testing a genuinely different part of the pipeline (the triggering decision) than the original test covered.

**Intermediate**

- Q: If this test fails, what are the two genuinely different places the failure could originate, and why does distinguishing them matter?
  A: It could fail at the triggering-decision layer (Topic 3's detection-and-trigger pattern doesn't recognize the invented product name), or at the downstream grounding layer (the catalog lookup or generation step, which Chapter 9 Topic 8 already validated, has since regressed). These require completely different fixes — refining the detection pattern versus investigating a regression in previously-working retrieval or generation logic — and correctly attributing which layer actually failed is what makes debugging efficient rather than guesswork.

- Q: Why shouldn't this single test be treated as comprehensive validation of triggering reliability in general?
  A: This test validates one specific, maximally-clean scenario extremely well, but real production queries include far more ambiguous, informal, or boundary cases than this one clean invented-product example covers. Chapter 10 Topic 7's broader, larger test suite is still necessary for genuinely comprehensive confidence — this topic's test is a strong point-validation, not a substitute for that broader coverage.

**Advanced**

- Q: Design the full, layered test this topic builds, explaining what each layer specifically validates and why testing them separately matters.
  A: Layer one tests Topic 3's `detect_named_product()` and `should_check_catalog()` functions directly against a query mentioning the invented product name, confirming the triggering decision fires correctly in isolation. Layer two, assuming layer one passes, tests whether the catalog lookup (or broader retrieval pipeline) correctly finds the invented product's specific, newly-added information. Layer three tests whether the final generated answer correctly cites those real, specific figures rather than a generic, hallucinated approximation — essentially re-running Chapter 9 Topic 8's original validation, but now as the downstream continuation of a properly-triggered agentic decision rather than an externally-forced retrieval call. Testing these layers separately means a failure at any specific point immediately reveals which part of the chain needs attention, rather than only knowing that "something" failed somewhere across the whole pipeline.

- Q: How would you decide whether this specific test's invented product name needs to be refreshed with a new one over time?
  A: Periodically verify the product name and its specific figures still don't produce a plausible-sounding confident answer from an ungrounded model call — if a model ever does start answering confidently and correctly about "Smart Saver FD" without any retrieval, this is a signal the name may have inadvertently entered training data (perhaps through this very test's own documentation being published or shared), and the test should be refreshed with a newly-invented, still-genuinely-novel product name and figures to preserve its clean, guaranteed-failure-mode property for future use.

**Scenario-based**

- Q: This test fails — the agent doesn't call `lookup_product_catalog` when the invented product name "Smart Saver FD" is mentioned. But when you manually call the catalog lookup with the exact same product name, it correctly finds the entry. Diagnose.
  A: Since the underlying catalog lookup itself works correctly when called directly, the failure is isolated specifically to the triggering-decision layer (Topic 3's detection pattern) — the agent's decision logic isn't correctly recognizing this specific product name as warranting a catalog check, despite the catalog itself having valid data for it. This points to refining `detect_named_product()`'s pattern-matching (perhaps the specific phrasing used in the failing test query doesn't match the detection logic's expected patterns) rather than investigating the catalog data or lookup function, which are already confirmed working correctly in isolation.

**Follow-up questions to expect:**

- "How would you extend this test to cover a case where the product is referenced with a genuine typo, not just an informal shorthand?"
- "What would you do if this test needed to validate triggering across multiple different novel products at once, rather than just Smart Saver FD?"


### 9. Hidden Concepts / Prerequisites Worth Knowing

- **This topic demonstrates a specific, valuable testing principle: reusing a well-designed test artifact (Chapter 9 Topic 8's invented product) to validate a genuinely different concern (this topic's triggering decision) rather than either retesting the same thing redundantly or designing an entirely new artifact from scratch** — recognizing when an existing test fixture can be extended to cover new ground, versus when a genuinely new fixture is needed, is a valuable, broadly-applicable testing discipline.
- **The layered testing approach this topic builds is a specific, concrete instance of the general "test each component in isolation before trusting the integrated whole" software testing principle** — unit-testing the triggering decision separately from integration-testing the full pipeline's behavior, exactly mirroring standard software testing discipline applied to this specific agentic RAG context.
- **This topic closes Chapter 13's complete arc**: Topic 1 established the conceptual static-vs-agentic distinction, Topic 2 established confidence-based triggering as the general principle, Topic 3 built one concrete, specific pattern implementing that principle, and this topic validates that specific pattern using the clearest, cleanest possible test case this notebook series has available — completing the chapter's full progression from concept through implementation to validated evidence.

### 10. Quick Revision Summary (for last-minute interview prep)

> This test completes what Chapter 9 Topic 8 explicitly deferred: that earlier test validated RAG's grounded-generation quality once retrieval was triggered, but assumed the triggering decision itself; this topic tests that triggering decision directly, using Topic 3's specific "check catalog only if uncertain" pattern applied to Chapter 9 Topic 8's exact invented "Smart Saver FD" product — genuinely absent from any model's training data, making a correct, unretrieved answer structurally impossible rather than merely unlikely. The test is deliberately layered: first confirming Topic 3's detection-and-trigger logic correctly identifies the invented product name as warranting a catalog check, then confirming (reusing Chapter 9 Topic 8's already-validated logic) that the catalog lookup and generation correctly find and cite the real, specific figures rather than a generic hallucination. This layering matters because a failure at either layer requires a genuinely different fix, and testing them together would only reveal that something failed, not where. This single, clean test validates one specific, maximally-clear scenario extremely well, but doesn't substitute for Chapter 10 Topic 7's broader test-suite coverage of more ambiguous, real-world triggering cases — and its own validity depends on the invented product name genuinely remaining absent from any training data indefinitely, worth periodically re-verifying rather than assuming forever.


### Module 1: Layer 1 — Testing Topic 3's Triggering Decision on the Exact Smart Saver FD Case

Reuse Topic 3's exact detection-and-trigger functions, unchanged, and test them specifically against Chapter 9 Topic 8's invented product query.

In [1]:

# ------------------------------------------------------------------
# MODULE 1: LAYER 1 -- triggering decision, tested in isolation
# (reusing Topic 3's EXACT functions, unchanged)
# ------------------------------------------------------------------

# the exact invented product from Chapter 9 Topic 8 -- genuinely
# absent from any real training data, invented specifically for testing
SMART_SAVER_FD_GROUND_TRUTH = {
    "product_name": "Smart Saver FD",
    "interest_rate_percent": 7.65,
    "tenure_months": 18,
    "minimum_deposit_inr": 25000,
}

PRODUCT_CATALOG = {
    "smart saver fd": {"interest_rate_percent": 7.65, "tenure_months": 18, "minimum_deposit_inr": 25000},
}


def detect_named_product(query: str) -> str:
    """EXACT SAME function from Topic 3, unchanged -- testing whether
    it also correctly handles THIS specific test case."""
    text_lower = query.lower()
    for product_name in PRODUCT_CATALOG.keys():
        if product_name in text_lower:
            return product_name
        name_words = product_name.replace(" fd", "").split()
        if all(word in text_lower for word in name_words) and len(name_words) > 1:
            return product_name
    return None


def should_check_catalog(query: str) -> bool:
    """EXACT SAME function from Topic 3, unchanged."""
    return detect_named_product(query) is not None


SMART_SAVER_TEST_QUERY = "What is the interest rate and tenure for the Smart Saver FD?"

print("=" * 70)
print("MODULE 1: LAYER 1 -- TRIGGERING DECISION, TESTED IN ISOLATION")
print("=" * 70)
print(f"Test query: '{SMART_SAVER_TEST_QUERY}'\n")

detected_product = detect_named_product(SMART_SAVER_TEST_QUERY)
triggering_decision = should_check_catalog(SMART_SAVER_TEST_QUERY)

print(f"Detected product: {detected_product}")
print(f"Should check catalog? {triggering_decision}")

layer_1_pass = triggering_decision is True
layer_1_label = "PASS" if layer_1_pass else "FAIL"
print(f"\nLAYER 1 RESULT: {layer_1_label}")
if layer_1_pass:
    print("The agent's OWN triggering decision correctly identified 'Smart Saver FD'")
    print("as a named product warranting a catalog check -- this is the specific")
    print("thing Chapter 9 Topic 8 explicitly did NOT test, since that test assumed")
    print("retrieval was already triggered.")

print("\nModule 1 complete. Run Module 2 next.")


MODULE 1: LAYER 1 -- TRIGGERING DECISION, TESTED IN ISOLATION
Test query: 'What is the interest rate and tenure for the Smart Saver FD?'

Detected product: smart saver fd
Should check catalog? True

LAYER 1 RESULT: PASS
The agent's OWN triggering decision correctly identified 'Smart Saver FD'
as a named product warranting a catalog check -- this is the specific
thing Chapter 9 Topic 8 explicitly did NOT test, since that test assumed
retrieval was already triggered.

Module 1 complete. Run Module 2 next.


### Module 2: Layer 2 — The Ungrounded Failure Mode, Reproduced

Confirm the ungrounded model MUST fail for this invented product, exactly reproducing Chapter 9 Topic 8's core argument -- this is what makes Layer 1's correct triggering actually matter.

In [2]:

# ------------------------------------------------------------------
# MODULE 2: LAYER 2 -- the ungrounded failure mode, reproduced
# ------------------------------------------------------------------

import re

def extract_numbers(text: str) -> set:
    return set(re.findall(r'\d+(?:\.\d+)?', text))


def simulate_ungrounded_response(query: str) -> str:
    """Simulates what a model WOULD say without retrieval -- since
    Smart Saver FD's real terms genuinely never existed in any
    training data, this MUST be a fabrication, not just a possibility."""
    return "The Smart Saver FD typically offers around 7.0 percent for a 24-month tenure."


ungrounded_answer = simulate_ungrounded_response(SMART_SAVER_TEST_QUERY)
ungrounded_numbers = extract_numbers(ungrounded_answer)
ground_truth_numbers = {str(SMART_SAVER_FD_GROUND_TRUTH["interest_rate_percent"]),
                         str(SMART_SAVER_FD_GROUND_TRUTH["tenure_months"])}

print("=" * 70)
print("MODULE 2: LAYER 2 -- UNGROUNDED FAILURE MODE (WHY LAYER 1 MATTERS)")
print("=" * 70)
print(f"Ungrounded answer: '{ungrounded_answer}'")
print(f"Numbers stated: {ungrounded_numbers}")
print(f"Actual correct numbers: {ground_truth_numbers}")

matches = ungrounded_numbers & ground_truth_numbers
print(f"Correct numbers matched: {matches if matches else 'NONE'}")

layer_2_confirms_risk = len(matches) == 0
print(f"\nLAYER 2 RESULT: risk confirmed = {layer_2_confirms_risk}")
if layer_2_confirms_risk:
    print("CONFIRMED: without Layer 1's correct triggering decision, this EXACT")
    print("query would produce a confident, WRONG answer -- this is precisely")
    print("why testing the triggering decision (Layer 1), not just downstream")
    print("grounding quality, is essential for this specific failure pattern.")

print("\nModule 2 complete. Run Module 3 next.")


MODULE 2: LAYER 2 -- UNGROUNDED FAILURE MODE (WHY LAYER 1 MATTERS)
Ungrounded answer: 'The Smart Saver FD typically offers around 7.0 percent for a 24-month tenure.'
Numbers stated: {'24', '7.0'}
Actual correct numbers: {'7.65', '18'}
Correct numbers matched: NONE

LAYER 2 RESULT: risk confirmed = True
CONFIRMED: without Layer 1's correct triggering decision, this EXACT
query would produce a confident, WRONG answer -- this is precisely
why testing the triggering decision (Layer 1), not just downstream
grounding quality, is essential for this specific failure pattern.

Module 2 complete. Run Module 3 next.


### Module 3: Layer 3 — The Full Chain, End to End

Connect Layer 1's triggering decision to a real catalog lookup and grounded response, confirming the complete pipeline (decision -> lookup -> correct answer) works together, not just each piece in isolation.

In [3]:

# ------------------------------------------------------------------
# MODULE 3: LAYER 3 -- the FULL chain, end to end
# ------------------------------------------------------------------

def lookup_product_catalog(product_name: str) -> dict:
    """Chapter 10 Topic 6's real design template -- structured output,
    exists/not-exists distinction, input echoed back."""
    key = product_name.strip().lower()
    product = PRODUCT_CATALOG.get(key)
    if product is None:
        return {"product_name": product_name, "exists": False}
    return {"product_name": product_name, "exists": True, **product}


def full_agentic_pipeline(query: str) -> dict:
    """The COMPLETE chain: triggering decision (Layer 1) -> catalog
    lookup (if triggered) -> grounded answer construction. This is
    what Chapter 9 Topic 8 explicitly did NOT test end-to-end, since
    it assumed the first step."""
    detected = detect_named_product(query)
    should_check = should_check_catalog(query)

    if not should_check:
        return {"triggered": False, "answer": "answered from general knowledge (no product detected)"}

    catalog_result = lookup_product_catalog(detected)
    if not catalog_result["exists"]:
        return {"triggered": True, "answer": f"'{detected}' not found in catalog"}

    rate = catalog_result["interest_rate_percent"]
    tenure = catalog_result["tenure_months"]
    grounded_answer = f"The {detected} offers {rate} percent interest for a {tenure} month tenure."
    return {"triggered": True, "catalog_result": catalog_result, "answer": grounded_answer}


print("=" * 70)
print("MODULE 3: LAYER 3 -- FULL CHAIN, END TO END")
print("=" * 70)
print(f"Query: '{SMART_SAVER_TEST_QUERY}'\n")

pipeline_result = full_agentic_pipeline(SMART_SAVER_TEST_QUERY)
print(f"Triggered catalog check? {pipeline_result['triggered']}")
print(f"Final answer: '{pipeline_result['answer']}'")

final_numbers = extract_numbers(pipeline_result["answer"])
final_matches = ground_truth_numbers & final_numbers
full_chain_correct = final_matches == ground_truth_numbers

print(f"\nNumbers in final answer: {final_numbers}")
print(f"Ground truth numbers: {ground_truth_numbers}")
print(f"ALL correct numbers present: {full_chain_correct}")

print("\n" + "=" * 70)
print("COMPLETE LAYERED TEST RESULT")
print("=" * 70)
print(f"  Layer 1 (triggering decision):     {'PASS' if layer_1_pass else 'FAIL'}")
print(f"  Layer 2 (ungrounded risk confirmed): {'PASS' if layer_2_confirms_risk else 'FAIL'}")
print(f"  Layer 3 (full chain, correct answer): {'PASS' if full_chain_correct else 'FAIL'}")

if layer_1_pass and layer_2_confirms_risk and full_chain_correct:
    print("\nALL THREE LAYERS PASS. This is the complete validation Chapter 9")
    print("Topic 8 explicitly deferred: not just 'does RAG produce a correct")
    print("answer once triggered' (already proven there), but 'does the agent")
    print("correctly DECIDE to trigger it in the first place' for the exact")
    print("new-product-name failure pattern this project cares about.")

print("\nModule 3 complete. All Chapter 13 Topic 4 modules done.")
print()
print("=" * 70)
print("CHAPTER 13 TOPIC 4 -- KEY POINTS TO REMEMBER")
print("=" * 70)
print("""
  This test completes what Chapter 9 Topic 8 explicitly deferred:
  testing the TRIGGERING DECISION itself, not just downstream grounded-
  generation quality (which that earlier test already validated).

  LAYERED testing (3 distinct layers, each independently checked) gives
  SPECIFIC diagnostic value if something fails -- demonstrated
  concretely: each layer has its own pass/fail result.

  Reuses Chapter 9 Topic 8's EXACT invented product and Topic 3's EXACT
  triggering functions, unchanged -- testing a genuinely different
  concern (the decision) using already-validated artifacts.

  This ONE clean test validates ONE specific scenario extremely well --
  it does NOT substitute for Chapter 10 Topic 7's broader test-suite
  coverage of messier, more ambiguous real-world triggering cases.

  This closes Chapter 13's full arc: static vs agentic (Topic 1),
  confidence-based triggering principle (Topic 2), one concrete
  pattern (Topic 3), and this pattern's validated proof (Topic 4).
""")


MODULE 3: LAYER 3 -- FULL CHAIN, END TO END
Query: 'What is the interest rate and tenure for the Smart Saver FD?'

Triggered catalog check? True
Final answer: 'The smart saver fd offers 7.65 percent interest for a 18 month tenure.'

Numbers in final answer: {'7.65', '18'}
Ground truth numbers: {'7.65', '18'}
ALL correct numbers present: True

COMPLETE LAYERED TEST RESULT
  Layer 1 (triggering decision):     PASS
  Layer 2 (ungrounded risk confirmed): PASS
  Layer 3 (full chain, correct answer): PASS

ALL THREE LAYERS PASS. This is the complete validation Chapter 9
Topic 8 explicitly deferred: not just 'does RAG produce a correct
answer once triggered' (already proven there), but 'does the agent
correctly DECIDE to trigger it in the first place' for the exact
new-product-name failure pattern this project cares about.

Module 3 complete. All Chapter 13 Topic 4 modules done.

CHAPTER 13 TOPIC 4 -- KEY POINTS TO REMEMBER

  This test completes what Chapter 9 Topic 8 explicitly deferred:
  